# Connection
Creates the handler responsible for communicating with the attached bear motors

In [52]:
import time
import math
import numpy as np
from typing import NamedTuple
from pybear import Manager

# Edit these if your setup differs from the defaults.
PORT = '/dev/serial/by-id/usb-Westwood_Robotics_USB2BEAR_Boosted_UB0104-if00-port0'
BAUD = 8000000  # USB2BEAR fast setting

# Constructing the BEAR object opens the serial port immediately.
# single_timeout / bulk_timeout default to 2 ms, single_try_num / bulk_try_num to 2.
bear = Manager.BEAR(
    port=PORT,
    baudrate=BAUD,
    single_timeout=0.01,   # 10 ms — comfortably above FTDI worst case
    single_try_num=3,
    bulk_timeout=0.01,
    bulk_try_num=3,
)

   __        __          _                               _    
   \ \      / /___  ___ | |_ __      __ ___    ___    __| |   
    \ \ /\ / // _ \/ __|| __|\ \ /\ / // _ \  / _ \  / _` |   
     \ V  V /|  __/\__ \| |_  \ V  V /| (_) || (_) || (_| |   
      \_/\_/  \___||___/ \__|  \_/\_/  \___/  \___/  \__,_|   
              ____         _             _    _               
             |  _ \  ___  | |__    ___  | |_ (_)  ___  ___    
             | |_) |/ _ \ | '_ \  / _ \ | __|| | / __|/ __|   
             |  _ <| (_) || |_) || (_) || |_ | || (__ \__ \   
             |_| \_\___/ |_.__/  \___/  \__||_| \___||___/   
                                                              
=== PyBEAR by Westwood Robotics -- Last Updated 2025.07.29 ===



# Motor Management
Creates the variables for managing the different motors. Stored inside a NamedTuple

In [53]:
class Motor(NamedTuple):
    ID:              int
    ENABLE:          bool = False
    TORQUE_ENABLE:   int = 0
    MODE:            int = 0
    ERROR_BYTE:      int = None
    ERROR_MSG:       str = None
    BUS_VOLTAGE:     float = 0.0
    WINDING_TEMP:    float = 0.0
    POWERSTAGE_TEMP: float = 0.0
    IC_TEMP:         float = 0.0
    POSITION:        float = 0.0

from enum import Enum

class Joint(Enum):
    HIP   = 1      # value doubles as the BEAR bus ID
    KNEE  = 2
    ANKLE = 3

class Mode(Enum):
    TORQUE   = 0
    VELOCITY = 1
    POSITION = 2
    FORCE    = 3   # "Direct Force" — combined position/velocity/torque

motors = {j: Motor(ID=j.value) for j in Joint}

# Access each motor with following syntax style
# motors[Joint.HIP]
# You can get Enum names and values as Joint.HIP.name (which is "HIP") and Joint.HIP.value (which is 1)

### Pings each motor for responsiveness

In [70]:
def update_status(key):
    motor = motors[key]
    ping_data, err = bear.ping(motor.ID)[0]
    if err is None:
        print(f"Error: Motor {motor.ID} is not responding...")
        motors[key] = motor._replace(ENABLE=False, ERROR_BYTE=None, ERROR_MSG="No Connection")
        return False
    else:
        print(f"BEAR {motor.ID} online. fw/hw version blob = {ping_data[0]:#x}")
        print(f"Error byte = 0x{err:02x} ({bear.decode_error(err)})")
        motors[key] = motor._replace(ENABLE=True, ERROR_BYTE=err, ERROR_MSG=f"{bear.decode_error(err)}")
        print(motors[key])
        return True

for key in motors.keys():
    update_status(key)

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
BEAR 1 online. fw/hw version blob = 0x503046f
Error byte = 0x80 (No Error!)
Motor(ID=1, ENABLE=True, TORQUE_ENABLE=0, MODE=2, ERROR_BYTE=128, ERROR_MSG='No Error!', BUS_VOLTAGE=24.65695571899414, WINDING_TEMP=32.25, POWERSTAGE_TEMP=34.9453125, IC_TEMP=36.8125, POSITION=1.3915667533874512)
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
BEAR 2 online. fw/hw version blob = 0x503046f
Error byte = 0x80 (No Error!)
Motor(ID=2, ENABLE=True, TORQUE_ENABLE=0, MODE=2, ERROR_BYTE=128, ERROR_MSG='No Error!', BUS_VOLTAGE=24.533945083618164, WINDING_TEMP=32.75, POWERSTAGE_TEMP=35.3828125, IC_TEMP=37.1875, POSITION=0.8424676656723022)
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
BEAR 3 online. fw/hw version blob = 0x503046f
Error byte = 0x80 (No Error!)
Motor(ID=3, ENABLE=True, TORQUE_ENABLE=0, MODE=2, ERROR_BYTE=128, ERROR_MSG='No Error!', BU

### Function for updating motor state

In [55]:
def data_read(getter, MOTOR_ID):
    """Convenience: flatten [([val], err)] -> (val, err) for single-motor single-reg reads."""
    data_list, err = getter(MOTOR_ID)[0]
    return data_list[0], err

def update_motor_states(key):
    motor = motors[key]

    v_bus, _     = data_read(bear.get_input_voltage, motor.ID)
    t_wind, _    = data_read(bear.get_winding_temperature, motor.ID)
    t_power, _   = data_read(bear.get_powerstage_temperature, motor.ID)
    t_ic, _      = data_read(bear.get_ic_temperature, motor.ID)
    pos, _       = data_read(bear.get_present_position, motor.ID)
    torque_en, _ = data_read(bear.get_torque_enable, motor.ID)
    mode, _      = data_read(bear.get_mode, motor.ID)

    motors[key] = motor._replace(BUS_VOLTAGE = v_bus,
                                 WINDING_TEMP = t_wind,
                                 POWERSTAGE_TEMP = t_power,
                                 IC_TEMP = t_ic,
                                 POSITION = pos,
                                 TORQUE_ENABLE = torque_en,
                                 MODE = mode)
    
for key in motors.keys():
    update_motor_states(key)
    print(motors[key])

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
Motor(ID=1, ENABLE=True, TORQUE_ENABLE=1, MODE=2, ERROR_BYTE=128, ERROR_MSG='No Error!', BUS_VOLTAGE=24.667436599731445, WINDING_TEMP=32.0, POWERSTAGE_TEMP=40.9453125, IC_TEMP=39.4375, POSITION=1.3459162712097168)
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single r

### Function for setting motor gain values and limits

In [ ]:
def set_motor_gains(key):
    motor = motors[key]
    
    # Make absolutely sure the motor is off before touching gains/limits.
    bear.set_torque_enable((motor.ID, 0))

    # Tune position-loop gains. These work for many KBMB02 setups; reduce if oscillation appears.
    bear.set_p_gain_position((motor.ID, 5.0))
    bear.set_i_gain_position((motor.ID, 0.01))
    bear.set_d_gain_position((motor.ID, 0.2))

    # Velocity loop gets used by position mode internally — must also be sane.
    bear.set_p_gain_velocity((motor.ID, 1.0))
    bear.set_i_gain_velocity((motor.ID, 0.0))
    bear.set_d_gain_velocity((motor.ID, 0.0))

    # --- Current loop (id and iq) — recommended for Koala Muscle Build V2 ---
    bear.set_p_gain_id((motor.ID, 0.358))
    bear.set_i_gain_id((motor.ID, 0.045))
    bear.set_d_gain_id((motor.ID, 0.0))
    bear.set_p_gain_iq((motor.ID, 0.358))
    bear.set_i_gain_iq((motor.ID, 0.045))
    bear.set_d_gain_iq((motor.ID, 0.0))

    # Spring-damper gains. P = stiffness, D = damping.
    bear.set_p_gain_force((motor.ID, 5.0))
    bear.set_i_gain_force((motor.ID, 0.0))   # I gain almost always causes oscillation here.
    bear.set_d_gain_force((motor.ID, 0.2))

    update_motor_states(key)

def set_motor_limits(key):
    motor = motors[key]

    # --- Safety limits ---
    # limit_i_max caps both id and iq. KBMB02 KT = 1.16 Nm/A, so 1.5 A ≈ 1.74 Nm at the output.
    bear.set_limit_i_max((motor.ID, 5))

    # limit_velocity_max applies in Velocity and Position modes (NOT Direct Force).
    bear.set_limit_velocity_max((motor.ID, 5.0))
    
    # Position bounds in Position and Direct Force modes — going outside triggers internal damping.
    bear.set_limit_position_min((motor.ID, -3*math.pi))
    bear.set_limit_position_max((motor.ID,  3*math.pi))

    update_motor_states(key)

for key in motors.keys():
    set_motor_gains(key)
    set_motor_limits(key)
    update_motor_states(key)
    print(motors[key])

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try ag

### Function for disabling motors

In [57]:
def disable(key):
    motor = motors[key]
    """Idempotent. Safe to call any time."""
    bear.set_torque_enable((motor.ID, 0))

Disable All Motors

In [153]:
for key in motors.keys():
    disable(key)

Cleanup 

In [154]:
def cleanup(motors_dict):
    for key in motors_dict.keys():
        motor = motors_dict[key]
        bear.set_torque_enable((motor.ID, 0))
    bear.close_port()
    print("Disabled and port closed.")

cleanup(motors)

Disabled and port closed.


### Functions for switching motor mode

In [60]:
def enable_position_mode(key):
    motor = motors[key]
    disable(key)
    time.sleep(0.01)  # let the motor coast / damping kick in before we rewrite goals

    pos, _ = data_read(bear.get_present_position, motor.ID)
    bear.set_goal_position((motor.ID, pos)) # set the goal position to current position
    
    bear.set_mode((motor.ID, Mode.POSITION.value))
    bear.set_torque_enable((motor.ID, 1))

    update_motor_states(key)

def enable_velocity_mode(key):
    motor = motors[key]
    disable(key)
    time.sleep(0.01)  # let the motor coast / damping kick in before we rewrite goals

    bear.set_goal_velocity((motor.ID, 0.0)) # set the goal velocity to 0

    bear.set_mode((motor.ID, Mode.VELOCITY.value))
    bear.set_torque_enable((motor.ID, 1))

    update_motor_states(key)

def enable_torque_mode(key):
    motor = motors[key]
    disable(key)
    time.sleep(0.01)  # let the motor coast / damping kick in before we rewrite goals

    bear.set_goal_iq((motor.ID, 0.0)) # set the goal torque (electric current iq) to 0

    bear.set_mode((motor.ID, Mode.TORQUE.value))
    bear.set_torque_enable((motor.ID, 1))

    update_motor_states(key)

def enable_force_mode(key):
    motor = motors[key]
    disable(key)
    time.sleep(0.01)  # let the motor coast / damping kick in before we rewrite goals

    pos, _ = data_read(bear.get_present_position, motor.ID)
    bear.set_goal_position((motor.ID, pos)) # set the goal position to current position
    bear.set_goal_velocity((motor.ID, 0.0)) # set the goal velocity to 0
    bear.set_goal_iq((motor.ID, 0.0)) # set the goal torque (electric current iq) to 0

    bear.set_mode((motor.ID, Mode.FORCE.value))
    bear.set_torque_enable((motor.ID, 1))

    update_motor_states(key)

### Functions for setting mode values

In [61]:
def set_position(key, dr):
    update_motor_states(key)
    motor = motors[key]
    bear.set_goal_position((motor.ID, motor.POSITION + dr))

### Checking Mode Switches

Single Motor Position Testing

In [104]:
for key in motors.keys():
    enable_position_mode(key)
time.sleep(1)

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try ag

In [106]:
test_motor_key = Joint.KNEE
set_position(test_motor_key, -0.25)

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...


# Direct Force hold with gravity feed-forward

Position mode can't take a torque feed-forward (`goal_iq` is only writable in modes 0 and 3), so it has to *integrate* its way to the gravity-holding torque — that's the "sag, then slowly creep back to the setpoint" behavior. Direct Force mode (mode 3) fixes this: it's a virtual spring/damper about `goal_position` **plus** a `goal_iq` feed-forward we use to cancel gravity directly.

```
iq = p_gain_force·(goal_position − pos) + d_gain_force·(0 − vel) + goal_iq
```

`calibrate_gravity_iq` finds the holding current for a pose; `hold_force` ramps to a target and holds it with that feed-forward. Reminder: Direct Force **ignores `limit_velocity_max` and the position limits** — only `limit_i_max` still caps torque — so the goal is ramped, not stepped.

In [152]:
def calibrate_gravity_iq(key, test_iq, settle=2.0):
    """Hold the CURRENT pose in Direct Force with goal_iq = test_iq and report drift.

    Tune test_iq until drift ~ 0:
      drift < 0 (sagging) -> increase test_iq
      drift > 0 (rising)  -> decrease test_iq
    If raising test_iq makes the sag WORSE, the sign is flipped — negate it.
    test_iq must stay within limit_i_max (currently 5 A) or the command is ignored.
    """
    enable_force_mode(key)                       # FORCE mode, holding present pos, goal_iq=0
    motor = motors[key]
    start, _ = data_read(bear.get_present_position, motor.ID)

    bear.set_goal_position((motor.ID, start))    # anchor the spring where we are now
    bear.set_goal_iq((motor.ID, test_iq))
    time.sleep(settle)

    end, _ = data_read(bear.get_present_position, motor.ID)
    drift = end - start
    update_motor_states(key)
    verdict = "holds — use this" if abs(drift) < 0.01 else ("sagging — raise iq" if drift < 0 else "rising — lower iq")
    print(f"test_iq={test_iq:+.3f} A  start={start:+.4f}  end={end:+.4f}  drift={drift:+.4f} rad  -> {verdict}")
    return drift


def hold_force(key, target, gravity_iq=0.0, step=0.05, dt=0.02):
    """Ramp to `target` (rad) in Direct Force mode and hold there.

    `gravity_iq` is the feed-forward holding current from calibrate_gravity_iq().
    Stays ENABLED on exit — the spring + feed-forward are what keep it in place,
    so do NOT disable() if you want it to keep holding.
    Direct Force ignores limit_velocity_max and the position limits, so the
    goal_position is ramped (a far target in one step would be a hard yank).
    """
    enable_force_mode(key)                       # FORCE mode, holding present pos, goal_iq=0
    motor = motors[key]
    start, _ = data_read(bear.get_present_position, motor.ID)

    bear.set_goal_iq((motor.ID, gravity_iq))     # apply gravity comp before moving
    n = max(1, int(abs(target - start) / step))
    for i in range(1, n + 1):
        bear.set_goal_position((motor.ID, start + (target - start) * i / n))
        time.sleep(dt)

    update_motor_states(key)
    print(f"Holding {key.name} at goal {target:+.4f} rad (now {motors[key].POSITION:+.4f}), gravity_iq={gravity_iq:+.3f} A")

### 1. Calibrate the gravity feed-forward

Pick the joint and pose you want to hold, then sweep `test_iq` until `drift ≈ 0`. Start at `0.0` and step in ~0.1 A increments; whatever value zeros the drift is your `gravity_iq` for that pose. The spring (`p_gain_force`) holds the limb while you tune, so it's safe to sweep. For a real leg the gravity torque changes with angle — calibrate near the pose you'll actually hold, or scale it by `cos(joint_angle)`.

In [107]:
cal_key = Joint.KNEE
# Sweep test_iq until drift ~ 0: try 0.0, then 0.1, 0.2, ... (negate if raising it makes the sag worse).
calibrate_gravity_iq(cal_key, test_iq=-0.9)

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try ag

-0.003417104482650757

### 2. Hold a target position

Once you know `gravity_iq` for the pose, command an absolute target. It ramps there in Direct Force and holds with the feed-forward — no sag-and-creep. Leave it enabled to keep holding; run the Disable / Cleanup cells above when done.

In [151]:
hold_key = Joint.KNEE
GRAVITY_IQ = -0.9   # <- paste the value you found with calibrate_gravity_iq above

update_motor_states(hold_key)
# Default target = current position, i.e. "hold right here" with gravity comp.
hold_force(hold_key, target=motors[hold_key].POSITION - 0.25, gravity_iq=GRAVITY_IQ)
# To move somewhere instead:
# hold_force(hold_key, target=0.5, gravity_iq=GRAVITY_IQ)

[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
[PyBEAR | WARNING] :: Single read timed out. Now try again ...
Holding KNEE at goal -0.1601 rad (now -0.1439), gravity